# От регистра до транзакции памяти

**Вопрос.** Какие ограничения возникают между выражением `y[i] = a*x[i] + b[i]` и результатом в памяти GPU?

**Ожидаемый результат.** Мы вручную исполним загрузки/FMA/сохранение, оценим занятость по конечному регистровому файлу, посчитаем транзакции памяти группы потоков и выведем выигрыш плиточного умножения. Базовая работа использует только Python; отдельный аппаратный проход находится в `labs/15_cuda_path`.

In [ ]:
import math
import platform
import random
import sys

random.seed(14)
print({"python": sys.version.split()[0], "platform": platform.platform()})

## 1. Исполняем пять смысловых инструкций

Память адресуется по именам только для наглядности. Арифметическая инструкция получает значения из регистров, а не непосредственно из словаря памяти.

In [ ]:
def execute_scalar(memory):
    registers = {}
    trace = []

    registers["r0"] = memory["x"]
    trace.append(("load x", dict(registers)))
    registers["r1"] = memory["a"]
    trace.append(("load a", dict(registers)))
    registers["r2"] = memory["b"]
    trace.append(("load b", dict(registers)))
    registers["r3"] = registers["r1"] * registers["r0"] + registers["r2"]
    trace.append(("fma", dict(registers)))
    memory["y"] = registers["r3"]
    trace.append(("store y", dict(registers)))
    return trace

memory = {"x": 2.0, "a": 3.0, "b": 4.0}
trace = execute_scalar(memory)
assert memory["y"] == 10.0
for operation, state in trace:
    print(f"{operation:8s} -> {state}")

## 2. Конечный регистровый файл ограничивает число активных блоков

Функция ниже — учебная верхняя оценка. Реальное устройство выделяет ресурсы порциями и имеет дополнительные ограничения, которые учитывает калькулятор занятости.

In [ ]:
DEVICE = {
    "registers_per_sm": 65_536,
    "shared_bytes_per_sm": 64 * 1024,
    "max_threads_per_sm": 2_048,
    "max_warps_per_sm": 64,
    "max_blocks_per_sm": 32,
}

def occupancy_estimate(registers_per_thread, threads_per_block=256, shared_per_block=0):
    warps_per_block = math.ceil(threads_per_block / 32)
    by_registers = DEVICE["registers_per_sm"] // (registers_per_thread * threads_per_block)
    by_shared = (DEVICE["shared_bytes_per_sm"] // shared_per_block
                 if shared_per_block else DEVICE["max_blocks_per_sm"])
    by_threads = DEVICE["max_threads_per_sm"] // threads_per_block
    by_warps = DEVICE["max_warps_per_sm"] // warps_per_block
    active_blocks = min(by_registers, by_shared, by_threads, by_warps, DEVICE["max_blocks_per_sm"])
    active_warps = active_blocks * warps_per_block
    return {
        "active_blocks": active_blocks,
        "active_warps": active_warps,
        "occupancy": active_warps / DEVICE["max_warps_per_sm"],
        "limits": {"registers": by_registers, "shared": by_shared,
                   "threads": by_threads, "warps": by_warps},
    }

cases = {r: occupancy_estimate(r) for r in (32, 64, 96, 128)}
assert cases[32]["active_blocks"] == 8
assert cases[96]["active_blocks"] == 2
for registers, result in cases.items():
    print(registers, result)

## 3. Считаем транзакции для адресов всей группы

Модель использует 32 потока, слова по 4 байта и участки транзакций по 32 байта. Эти числа соответствуют учебному случаю NVIDIA CC 6.0+, но функция оставляет их параметрами.

In [ ]:
def memory_segments(offset_words=0, stride_words=1, lanes=32,
                    word_bytes=4, segment_bytes=32):
    addresses = [(offset_words + lane * stride_words) * word_bytes for lane in range(lanes)]
    segments = sorted({address // segment_bytes for address in addresses})
    return addresses, segments

aligned = memory_segments()
shifted = memory_segments(offset_words=1)
assert len(aligned[1]) == 4
assert len(shifted[1]) == 5
for stride in (1, 2, 8, 32):
    _, segments = memory_segments(stride_words=stride)
    useful = 32 * 4
    transferred = len(segments) * 32
    print({"stride_words": stride, "transactions": len(segments),
           "useful_bytes": useful, "covered_bytes": transferred})

# Намеренно неверная модель считает отдельную транзакцию на поток.
def broken_transaction_count(lanes=32):
    return lanes

assert broken_transaction_count() != len(aligned[1])

## 4. Плитка уменьшает повторные чтения

Для одного произведения матриц 32×32 наивно считаем по 32 чтения из каждой матрицы на каждый из 1024 результатов. В учебной одноблочной плитке каждая исходная матрица загружается один раз.

In [ ]:
n = 32
naive_element_reads = n * n * (n + n)
tiled_element_reads = n * n + n * n
assert naive_element_reads // tiled_element_reads == 32
print({"naive_reads": naive_element_reads,
       "tiled_reads": tiled_element_reads,
       "reuse_factor": naive_element_reads / tiled_element_reads})

# Одна матрица BookLM: 256 x 768, BF16.
d_in, d_out, tokens = 256, 768, 512
weight_bytes = d_in * d_out * 2
decode_flops = 2 * d_in * d_out
decode_bytes = weight_bytes + d_in * 2 + d_out * 2
prefill_flops = tokens * decode_flops
prefill_bytes = weight_bytes + tokens * d_in * 2 + tokens * d_out * 2
decode_intensity = decode_flops / decode_bytes
prefill_intensity = prefill_flops / prefill_bytes
assert decode_intensity < 1.0
assert prefill_intensity > 100.0
print({"decode_FLOP_per_byte": round(decode_intensity, 3),
       "prefill_FLOP_per_byte": round(prefill_intensity, 3)})

## 5. Контракт смешанной точности не является одним флагом

In [ ]:
REQUIRED = ("storage", "instruction", "kernel", "runtime", "numerics")

def missing_evidence(report):
    return [field for field in REQUIRED if not report.get(field)]

w4a16_report = {
    "storage": "two signed 4-bit codes per byte + group scales",
    "instruction": "must be filled from disassembly or official ISA table",
    "kernel": "must be filled from profiler trace",
    "runtime": "same shapes, warm-up and selected device",
    "numerics": "max error and task metric versus BF16",
}
assert missing_evidence(w4a16_report) == []
broken_report = dict(w4a16_report, instruction="")
assert missing_evidence(broken_report) == ["instruction"]
print(w4a16_report)

## 6. Переход на настоящий GPU

Выполните команды из `labs/15_cuda_path/README.md` и добавьте к таблицам ноутбука: модель GPU, вычислительную возможность, версии драйвера и `nvcc`, регистры на поток, локальную память, активные группы, машинные инструкции и время после прогрева. Инварианты не меняются: результат AXPY равен 10; уменьшение регистров оценивается вместе с проливами; адреса группы объясняют транзакции; низкая точность считается подтверждённой только после заполнения всех пяти полей.